In [1]:
!git clone https://github.com/jjeong3150/AIFFEL_final_pjt_book-recommendation-agent

fatal: destination path 'AIFFEL_final_pjt_book-recommendation-agent' already exists and is not an empty directory.


In [2]:
!pip install -q qdrant-client
!pip install -q langchain
!pip install -q langchain-openai
!pip install -q langchain-naver

In [3]:
# 모듈 불러오기
%run /content/AIFFEL_final_project_peekabook/research/src/state/state_v1.ipynb
%run /content/AIFFEL_final_project_peekabook/research/src/db/qdrant.py
%run /content/AIFFEL_final_project_peekabook/research/src/embedding/embedder.py

In [4]:
import os
from google.colab import userdata

# Set the OpenAI API key environment variable
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Set the HCX API key environment variable
os.environ["CLOVASTUDIO_API_KEY"] = userdata.get('CLOVASTUDIO_API_KEY')

# Set the Qdrant API key environment variable
os.environ["QDRANT_API_KEY"] = userdata.get('QDRANT_API_KEY')
os.environ["QDRANT_URL"] = userdata.get('QDRANT_URL')

In [5]:
from langchain_openai import ChatOpenAI
from langchain_naver import ChatClovaX

# ── 초기화 ──────────────────────────────
embedder = LocalEmbedder("BAAI/bge-m3")   # 임베딩
db = QdrantDB(vector_size=1024)           # Qdrant 연결

# LLM
llm = ChatOpenAI(model="gpt-4o-mini")     # OpenAI LLM
# llm = ChatClovaX(model="HCX-005", temperature=0)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [17]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from qdrant_client.models import Filter, FieldCondition, MatchAny
import json


# 상위 카테고리 목록 (실제 cate_depth1 값으로 확인 후 수정)
CATEGORY_LIST = ["소설", "대학교재/전문서적", "어린이", "수험서/자격증", "시/에세이", "종교", "유아", "만화", "사회/정치", "경제/경영", "인문", "예술/대중문화", "국어/외국어", "고등학교 참고서", "자기계발", "초등학교 참고서", "건강/취미", "컴퓨터/IT", "역사", "자연/과학", "청소년", "중학교 참고서", "가정/요리", "여행", "잡지", "전집", "외국도서"]

# 장르 추출 노드
genre_prompt = ChatPromptTemplate.from_template("""
사용자 프로파일을 보고 아래 카테고리 목록에서 적합한 것을 최대 3개 선택하세요.
목록에 없는 값은 절대 반환하지 마세요.

카테고리 목록: {category_list}
사용자 프로파일: {summary}

JSON으로만 반환: {{"categories": ["소설"]}}
""")

def extract_genre_node(state: CRSState) -> dict:
    summary = state.get("summary", "")
    chain = genre_prompt | llm
    response = chain.invoke({
        "category_list": CATEGORY_LIST,
        "summary": summary
    })
    try:
        categories = json.loads(response.content)["categories"]
    except (json.JSONDecodeError, KeyError):
        categories = []  # 파싱 실패 시 필터 없이 진행

    print(categories)

    return {"genre_filter": categories}

In [18]:
def simple_rag_with_filter_node(state: CRSState) -> dict:
    # Qdrant에서 후보 책 5개 검색 -> retrieved_books 저장
    summary = state.get("summary", "")
    categories = state.get("genre_filter", [])
    query_vector = embedder.embed(summary)

    query_filter = None
    if categories:
        results = db.search_with_filter(
            "books_v1",
            query_vector,
            query_filter=Filter(
                must=[FieldCondition(
                    key="cate_depth1",
                    match=MatchAny(any=categories)
                )]
            ),
            limit=10,
            threshold=0.5,
        )
    else:
        results = db.search("books_v1", query_vector, limit=10, threshold=0.5)

    # 다음 노드로 넘길 형태로 정리
    retrieved_books = [
        {
            "isbn": r.payload.get("isbn"),
            "title": r.payload.get("title"),
            "author": r.payload.get("author"),
            "book_intro": r.payload.get("book_intro"),
            "cate_depth1": r.payload.get("cate_depth1")
        }
        for r in results
    ]

    return {"retrieved_books": retrieved_books}


In [23]:

def rag_llm_node(state: CRSState) -> dict:
    summary = state.get("summary", "")
    books = state["retrieved_books"]

    context = "\n\n".join([
        f"ISBN: {b['isbn']}\n"
        f"제목: {b['title']}\n"
        f"저자: {b['author']}\n"
        f"소개: {b['book_intro'][:100]}...\n"
        f"장르: {b['cate_depth1']}"
        for b in books
    ])

    # 프롬프트 직접 정의
    rag_prompt = ChatPromptTemplate.from_template("""
당신은 도서관 큐레이터 AI입니다.

[규칙]
- 반드시 [검색된 도서 목록]에 있는 책만 추천하세요.
- 반드시 JSON 형식으로만 답하세요. 다른 텍스트는 절대 포함하지 마세요.
- 사용자 프로파일을 참고해서 가장 적합한 도서 3권을 추천하세요.
- 추천 이유는 반드시 사용자 프로파일의 독서 목적, 성향, 상황과 연결해서 작성하세요.

[사용자 프로파일]
{summary}

[검색된 도서 목록]
{context}

[출력 형식]
[
    {{"title": "책 제목", "author": "저자", "isbn": "ISBN번호", "book_intro": "책 소개", "cate_depth1": "장르", "reason": "추천 이유 2~3문장"}},
    {{"title": "책 제목", "author": "저자", "isbn": "ISBN번호", "book_intro": "책 소개", "cate_depth1": "장르", "reason": "추천 이유 2~3문장"}},
    {{"title": "책 제목", "author": "저자", "isbn": "ISBN번호", "book_intro": "책 소개", "cate_depth1": "장르", "reason": "추천 이유 2~3문장"}}
]
"""
    )

    chain = rag_prompt | llm
    response = chain.invoke({
        "context": context,
        "summary": summary
    })

    try:
        recommendations = json.loads(response.content)
    except json.JSONDecodeError:
        # 파싱 실패해도 LLM 응답 그대로 반환
        recommendations = response.content

    # recommendations 출력 확인
    # print(recommendations)

    return {
        "messages": [AIMessage(content=response.content)],
        "recommendations": recommendations
    }

In [24]:
# ── TEST 실행 ────────────────────────────────
graph_test = StateGraph(CRSState)
graph_test.add_node("genre", extract_genre_node)  # 장르 추출 노드 추가
graph_test.add_node("rag",   simple_rag_with_filter_node)
graph_test.add_node("llm",   rag_llm_node)
graph_test.add_edge(START,   "genre")
graph_test.add_edge("genre", "rag")
graph_test.add_edge("rag",   "llm")
graph_test.add_edge("llm",   END)
app_test = graph_test.compile()

# TEST_SUMMARY = "사용자는 한국 현대소설을 선호하며, 감성을 풍부하게 하고 다양한 시각을 배우고자 하는 목표를 가지고 있습니다. 현재 감성적이고 몽환적인 기분 속에서 잔잔한 음악과 함께 조용한 공간에서 독서를 즐기고 있으며, 스토리와 감정에 집중하는 스타일을 선호합니다. 중급 난이도의 작품을 통해 감정 정리와 아름다움을 느끼고자 하는 상황입니다."
TEST_SUMMARY = "사용자는 주말에 가볍게 읽으면서 스트레스를 해소하고 새로운 아이디어를 얻고자 하며, SF와 테크 스릴러 장르의 흥미로운 이야기를 선호합니다. 독서 스타일은 빠르게 전체적인 흐름을 파악하고, 스토리의 흡입력에 집중하는 중급자용입니다. 깊이 있는 분석보다는 이야기 전개와 흥미로운 설정에 중점을 두고 있으며, 가벼운 책을 통해 재미를 추구하고 있습니다."

result_test = app_test.invoke({
    "messages": [HumanMessage(content=TEST_SUMMARY)],
    "summary": TEST_SUMMARY,
    "retrieved_books": [],
    "recommendations": [],
    "cate_depth1": []
})

print(result_test["messages"][-1].content)

['소설']
[
    {
        "title": "당신의 머리 위에 1",
        "author": "박건",
        "isbn": "9791104912092",
        "book_intro": "SF소설은 대개 어렵고, 지루하고, 따분할 것이라는 인식이 짙다. 하지만 당신의 머리 위에 는 그 이미지를 과감하게 반전시킨다. 관대하는 평범한 삶을 꿈꾸는 고등학생이지만 그 바람...",
        "cate_depth1": "소설",
        "reason": "이 책은 SF 장르로서 흥미로운 이야기 전개를 가지고 있어, 가벼운 주말 독서에 적합합니다. 또한 빠르게 전개되는 스토리에 집중할 수 있어 독서 스타일에 잘 맞습니다."
    },
    {
        "title": "숨겨진 얼굴",
        "author": "이현종",
        "isbn": "9791190408752",
        "book_intro": "스릴러물의 가장 큰 매력은 다양한 소재와 반전이다. 새로운 작가가 독자의 기대를 가득 채워 줄 새로운 소재, 새로운 스토리텔링! 누군가가 부모를 죽였다! 사랑하는 가족을 잃은 건 ...",
        "cate_depth1": "소설",
        "reason": "반전이 매력적인 스릴러 장르로, 빠른 전개와 흥미로운 설정이 가득하여 스트레스를 해소하는 데 도움을 줄 것입니다. 긴장감 넘치는 이야기는 중급자 독자인 당신의 집중력을 끌어올립니다."
    },
    {
        "title": "블루스카이 3 - 피로 얼룩진 혁명군",
        "author": "박성재",
        "isbn": "9788990019899",
        "book_intro": "탄탄한 구성이 돋보이는 소프트 어드벤처 판타지의 모델. 이제 판타지가 인류의 일그러진 미래로 눈을 돌렸다. 이 소설은 실패한 인류의 미래를 구원할 실낱같은 희망을 찾아나가고 있다....",
     

In [10]:
# result["recommendations"]